<a href="https://colab.research.google.com/github/NikoriakViktot/PY-Course-Victor-Nikoriak-22-09-2026/blob/main/module_1/lessons/lesson_07_functions/note_lesson_07_functions.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Урок 7 — Функції

> За цей урок: замість того щоб тримати всю програму одним суцільним блоком (Урок 6 — звіт кафе за днями), навчаєшся розбивати її на іменовані, багаторазові шматки — `def`, параметри, `return`, декомпозиція. Головна вправа уроку — **рефакторинг** звіту кафе з уроку 6 у функції, крок за кроком, з перевіркою на кожному кроці.

Структура уроку: **RETRIEVE → CONCEPT → PREDICT/RUN/INVESTIGATE/MODIFY → CREATE → TRANSFER** (та сама послідовність, що й в Уроці 4).

## 🔁 RETRIEVE — пригадай Урок 6 (без підглядання)

Урок 6 був про цикли, словники та comprehensions. Дай відповідь усно чи на папері, **не запускаючи нічого**:

1. Що поверне `for k, v in {'a': 1}:`  — спрацює нормально чи впаде з помилкою? Якщо впаде — з якою?
2. Як одним рядком (comprehension) отримати список квадратів **парних** чисел від 1 до 10?
3. `d.setdefault('x', []).append(5)` — що робить цей рядок, якщо ключа `'x'` в `d` ще немає?

Звір відповіді нижче.

<details>
<summary>Відповіді</summary>

1. <code>ValueError</code> — <code>for k, v in d</code> (без <code>.items()</code>) ітерує по <b>ключах</b>, а ключ <code>'a'</code> — рядок з одного символу, який не можна розпакувати у дві змінні <code>k, v</code>.
2. <code>[x**2 for x in range(1, 11) if x % 2 == 0]</code>
3. Створює <code>d['x'] = []</code>, а потім одразу додає до нього <code>5</code> — після рядка <code>d == {'x': [5]}</code>.

</details>

## 📖 CONCEPT

### 1. Навіщо функції?

Той самий код, повторений кілька разів — проблема: якщо треба виправити логіку, доведеться шукати й правити **усі** копії.

In [ ]:
# БЕЗ функції — та сама формула переказана тричі
balance1 = 1000
interest1 = balance1 * 0.05
balance1_after = balance1 + interest1

balance2 = 2500
interest2 = balance2 * 0.05   # та сама формула — дублювання
balance2_after = balance2 + interest2

print(f"БЕЗ функції: {balance1_after}, {balance2_after}")

# З функцією — логіка в одному місці
def apply_interest(balance, rate=0.05):
    """Повертає баланс після нарахування відсотків."""
    return balance + balance * rate

print("З функцією:", apply_interest(1000), apply_interest(2500))
print("Інша ставка:", apply_interest(1000, rate=0.10))

### 2. Синтаксис: визначення, параметри, `return` vs `print`

```
def  назва(param1, param2=значення_за_замовчуванням):   ← сигнатура
    тіло функції
    return результат                                     ← ОБОВ'ЯЗКОВО return, якщо
                                                             результат потрібен ДАЛІ в коді
```

- **Параметр** — ім'я в дужках при *визначенні* (`def f(x):`). **Аргумент** — реальне значення при *виклику* (`f(5)`).
- Параметр зі значенням за замовчуванням (`rate=0.05`) можна не вказувати при виклику.
- `print()` показує значення **людині** на екрані. `return` передає значення **назад у програму**, щоб зберегти в змінну й використати далі. Функція без `return` завжди повертає `None`.

In [ ]:
# print замість return — типова пастка
def bad_double(x):
    print(x * 2)     # тільки показує

def good_double(x):
    return x * 2      # повертає назад у програму

result_bad = bad_double(5)     # надрукує 10, але result_bad = None
result_good = good_double(5)   # нічого не друкує, result_good = 10

print("result_bad  =", result_bad)
print("result_good =", result_good)
print("result_good * 3 =", result_good * 3)   # можна використати далі

try:
    result_bad * 3
except TypeError as e:
    print("result_bad * 3 -> TypeError:", e)

### 3. Декомпозиція

**Принцип єдиної відповідальності:** одна функція — одна задача. Якщо функцію важко назвати одним дієсловом — вона, ймовірно, робить забагато.

Маленький бонус: функція, що не змінює нічого поза собою (не мутує вхідні дані, не читає/пише глобальні змінні) і при однакових аргументах завжди повертає однаковий результат, називається **чистою** — її найлегше тестувати і найбезпечніше повторно використовувати.

In [ ]:
# Декомпозиція: велика задача -> маленькі одноцільові функції
def is_low_stock(quantity, threshold=5):
    """Предикат: чи товару залишилось мало."""
    return quantity <= threshold


def restock_amount(quantity, target=20):
    """Трансформер: скільки треба замовити, щоб дійти до target."""
    return max(0, target - quantity)


def inventory_report(stock):
    """Декомпозиція: використовує обидві функції вище для повного звіту."""
    low = {item: qty for item, qty in stock.items() if is_low_stock(qty)}
    orders = {item: restock_amount(qty) for item, qty in low.items()}
    return {"low_stock": low, "to_order": orders}


stock = {"олівці": 3, "зошити": 40, "лінійки": 5, "ручки": 12}
report = inventory_report(stock)
print("Мало на складі:", report["low_stock"])
print("Замовити:", report["to_order"])

### 4. Три патерни: Predicate / Transformer / Reducer

| Патерн | Питання | Вхід → Вихід | Приклад |
|---|---|---|---|
| **Predicate** | Так/Ні? | 1 елемент → `bool` | `is_low_stock(qty)` |
| **Transformer** | Як змінити форму? | 1 елемент → 1 елемент (нової форми) | `restock_amount(qty)` |
| **Reducer** | Як зібрати в одне? | багато елементів → 1 значення | `sum(...)`, `max(...)` |

Ці три питання — універсальний спосіб підійти до будь-якої задачі обробки списку даних, і саме вони знадобляться нижче в CREATE і TRANSFER.

In [ ]:
# Три патерни разом на одному наборі даних
scores = [55, 82, 78, 92, 45, 88, 63]

def is_passed(score):          # PREDICATE
    return score >= 60

def to_letter(score):          # TRANSFORMER
    if score >= 90: return "A"
    if score >= 80: return "B"
    if score >= 70: return "C"
    return "D"

passed = [s for s in scores if is_passed(s)]         # PREDICATE у comprehension
letters = [to_letter(s) for s in passed]              # TRANSFORMER у comprehension
average = sum(passed) / len(passed)                    # REDUCER

print("Здали:", passed)
print("Оцінки:", letters)
print(f"Середня серед здали: {average:.1f}")

### Що свідомо НЕ увійшло в цей конспект

Формат уроку — приблизно 2 години, і більшість цього часу піде на керовану вправу нижче (рефакторинг звіту кафе). Тому частину тем свідомо **відкладено** — кожна з них повернеться пізніше за програмою курсу:

| Тема | Де з'явиться |
|---|---|
| **`*args` / `**kwargs`** — змінна кількість аргументів | Урок 9 — Декоратори: обгортка `wrapper(*args, **kwargs)` приймає функцію з будь-якою сигнатурою |
| повний **pipeline** Filter → Map → Reduce з генераторами та порівнянням пам'яті | Урок 10 — Ітератори й генератори; як патерн — Урок 18 — Функції як об'єкти першого класу |
| поглиблено: **чисті функції** (impure vs pure, `.sort()` vs `sorted()`), незмінність даних | Урок 18 — Функції як об'єкти першого класу (патерн «Pure Functions & Immutability») |
| детальний розбір **stack frame** | Практикум П4 (урок 22) — рекурсія, де без стеку викликів не обійтися. Тут — лише одна згадка про ізоляцію локальних змінних |
| `global` та області видимості | довідник [Простори імен / LEGB](https://nikoriakviktot.github.io/PY-Course-Victor-Nikoriak-22-09-2026/reference/python_core/namespaces_legb/) |

Повний конспект про функції — [`notes_functions.ipynb`](https://colab.research.google.com/github/NikoriakViktot/PY-Course-Victor-Nikoriak-22-09-2026/blob/main/module_1/lessons/lesson_07_functions/notes_functions.ipynb) у цій же папці уроку. Там усі ці теми зібрані разом, плюс каталог **типових помилок** і підсумкова **шпаргалка**. Він глибший за обсягом, ніж потрібно для цього уроку, — зручний для самостійного повторення.

## 🍽️ PREDICT / RUN / INVESTIGATE / MODIFY — рефакторинг звіту кафе

Головна вправа уроку. На Уроці 6 ми написали звіт кафе за днями: кількість чеків, виторг і середній чек для кожного дня, найкращий день, чеки за прийомом їжі. Він працює, але це один суцільний блок коду. Задача цього блоку — розкласти його на функції **крок за кроком** і на кожному кроці **перевірити**, що поведінка не зламалась, а не просто повірити на слово.

Спочатку — дані: ті самі п'ять чеків, що й в уроці 6.

In [ ]:
import io
import contextlib
from typing import NamedTuple


class Order(NamedTuple):
    total_bill: float
    tip: float
    day: str
    time: str
    size: int


orders = [
    Order(540.0, 50.0, "пт", "вечеря", 2),
    Order(320.0, 30.0, "пт", "обід", 1),
    Order(980.0, 120.0, "сб", "вечеря", 4),
    Order(760.0, 70.0, "сб", "вечеря", 3),
    Order(450.0, 0.0, "нд", "обід", 5),
]
print("Чеків:", len(orders))

### PREDICT

Нижче — звіт з уроку 6 без змін. Перш ніж запускати, дай відповідь:

1. Що доведеться зробити, щоб отримати такий самий звіт для **іншого** списку чеків, наприклад за наступний вікенд?
2. Які змінні потрібні лише як проміжні кроки, а які — для друку?
3. Як перевірити, що правильно рахується **лише** виторг за днями, не запускаючи весь звіт?

<details>
<summary>Відповідь</summary>

1. Скопіювати весь блок і замінити `orders` на новий список — усі тридцять рядків. Будь-яке виправлення потім доведеться робити в обох копіях.
2. `orders_by_day`, `revenue_by_day`, `bills_by_time`, `average_by_day`, `best_day` — усі вони лежать поруч в одному просторі імен, і будь-яка частина коду може випадково їх змінити або перезаписати.
3. Ніяк: виторг рахується всередині спільного циклу разом з іншими словниками. Щойно цей шматок стане функцією `revenue_by_day(orders)`, її можна буде викликати окремо на двох-трьох чеках, де відповідь відома заздалегідь.

</details>

### RUN

Запускаємо звіт «до» рефакторингу. Вивід не лише друкуємо, а й зберігаємо в змінну `output_before` (через `contextlib.redirect_stdout`) — наприкінці порівняємо з ним вивід версії «після».

In [ ]:
buffer_before = io.StringIO()
with contextlib.redirect_stdout(buffer_before):
    orders_by_day = {}
    revenue_by_day = {}
    bills_by_time = {}
    for order in orders:
        orders_by_day[order.day] = orders_by_day.get(order.day, 0) + 1
        revenue_by_day[order.day] = revenue_by_day.get(order.day, 0) + order.total_bill
        bills_by_time.setdefault(order.time, []).append(order.total_bill)

    average_by_day = {day: revenue_by_day[day] / orders_by_day[day] for day in revenue_by_day}

    best_day = None
    for day, revenue in revenue_by_day.items():
        if best_day is None or revenue > revenue_by_day[best_day]:
            best_day = day

    for day in revenue_by_day:
        print(day, "— чеків:", orders_by_day[day], "виторг:", revenue_by_day[day], "середній:", average_by_day[day])
    print("Найкращий день:", best_day)
    print("За прийомом їжі:", bills_by_time)

output_before = buffer_before.getvalue()
print(output_before)

### INVESTIGATE

У звіті видно кілька природних блоків, кожен із чітким входом і виходом — саме такі блоки й стають функціями:

| Блок логіки | Майбутня функція | Вхід → вихід |
|---|---|---|
| лічильник чеків за днями | `count_by_day(orders)` | чеки → `{день: кількість}` |
| виторг за днями | `revenue_by_day(orders)` | чеки → `{день: сума}` |
| середній чек | `average_by_day(counts, revenue)` | два словники → `{день: середнє}` |
| пошук лідера | `best_day(revenue)` | `{день: сума}` → день |
| групування за прийомом їжі | `bills_by_time(orders)` | чеки → `{прийом: [суми]}` |
| друк | `print_report(orders)` | чеки → текст на екрані |

Перші п'ять функцій лише **рахують і повертають** результат (`return`), нічого не друкуючи. Друкує тільки `print_report`. Нижче — чотири контрольні точки: кожна визначає функцію і одразу її перевіряє через `assert`, перш ніж рухатись далі.

> Зверни увагу: імена `revenue_by_day`, `best_day`, `bills_by_time`, `average_by_day` зараз зайняті змінними зі звіту «до». Коли ми оголосимо функції з цими іменами, `def` перезапише старі змінні — це нормально, «до»-версія вже відпрацювала, а її вивід збережено в `output_before`.

### MODIFY — Контрольна точка 1: `count_by_day` (розібраний приклад)

Перша функція вже готова — прочитай і запусти, нічого міняти не треба. Словник створюється **всередині** функції, заповнюється циклом і повертається через `return`.

In [ ]:
def count_by_day(orders):
    """Кількість чеків у кожен день."""
    counts = {}
    for order in orders:
        counts[order.day] = counts.get(order.day, 0) + 1
    return counts


print(count_by_day(orders))

assert count_by_day(orders) == {"пт": 2, "сб": 2, "нд": 1}
assert count_by_day([]) == {}
assert count_by_day([Order(100.0, 0.0, "пн", "обід", 1)]) == {"пн": 1}
print("OK — count_by_day працює і на порожньому списку")

### Контрольна точка 2: `revenue_by_day` (заповни пропуск)

Та сама схема, що й у `count_by_day`, але замість `+ 1` додаємо суму чека.

In [ ]:
def revenue_by_day(orders):
    """Сума чеків за кожен день."""
    revenue = {}
    # TODO: пройдись по orders і додай order.total_bill до revenue[order.day]
    # BEGIN SOLUTION
    for order in orders:
        revenue[order.day] = revenue.get(order.day, 0) + order.total_bill
    # END SOLUTION
    return revenue


print(revenue_by_day(orders))

assert revenue_by_day(orders) == {"пт": 860.0, "сб": 1740.0, "нд": 450.0}
assert revenue_by_day([]) == {}
print("OK")

### Контрольна точка 3: `average_by_day` + `best_day` (заповни пропуски)

`average_by_day` не проходить чеки заново — вона отримує два вже готові словники. `best_day` — пошук лідера з уроку 6, але тепер `best` — локальна змінна, а результат повертається.

In [ ]:
def average_by_day(counts, revenue):
    """Середній чек за кожен день."""
    # BEGIN SOLUTION
    return {day: revenue[day] / counts[day] for day in revenue}
    # END SOLUTION


def best_day(revenue):
    """День з найбільшим виторгом (None для порожнього словника)."""
    best = None
    # BEGIN SOLUTION
    for day, amount in revenue.items():
        if best is None or amount > revenue[best]:
            best = day
    # END SOLUTION
    return best


counts = count_by_day(orders)
revenue = revenue_by_day(orders)
print(average_by_day(counts, revenue))
print(best_day(revenue))

assert average_by_day(counts, revenue) == {"пт": 430.0, "сб": 870.0, "нд": 450.0}
assert best_day(revenue) == "сб"
assert best_day({"пт": 100.0, "сб": 300.0, "нд": 200.0}) == "сб"
assert best_day({}) is None
print("OK")

### Контрольна точка 4: `bills_by_time` (заповни пропуск)

Групування з уроку 6: `setdefault(ключ, [])` створює порожній список для нового ключа, а `.append(...)` додає до нього суму чека.

In [ ]:
def bills_by_time(orders):
    """Суми чеків, згруповані за прийомом їжі."""
    groups = {}
    # BEGIN SOLUTION
    for order in orders:
        groups.setdefault(order.time, []).append(order.total_bill)
    # END SOLUTION
    return groups


print(bills_by_time(orders))

assert bills_by_time(orders) == {"вечеря": [540.0, 980.0, 760.0], "обід": [320.0, 450.0]}
assert bills_by_time([]) == {}
print("OK")

### Збірка: звіт з функцій

Усі функції готові — тепер складаємо з них звіт. `print_report` — єдина функція, що друкує: вона викликає решту і показує їхні результати. Її тіло читається як перелік кроків звіту.

In [ ]:
def print_report(orders):
    """Друкує звіт кафе за списком чеків."""
    counts = count_by_day(orders)
    revenue = revenue_by_day(orders)
    average = average_by_day(counts, revenue)
    for day in revenue:
        print(day, "— чеків:", counts[day], "виторг:", revenue[day], "середній:", average[day])
    print("Найкращий день:", best_day(revenue))
    print("За прийомом їжі:", bills_by_time(orders))


print("print_report визначено")

### Найважливіша перевірка: «до» і «після» дають ІДЕНТИЧНИЙ вивід

Запускаємо `print_report(orders)` з тими самими чеками і порівнюємо вивід з `output_before` **символ у символ**.

In [ ]:
buffer_after = io.StringIO()
with contextlib.redirect_stdout(buffer_after):
    print_report(orders)

output_after = buffer_after.getvalue()
print(output_after)

assert output_after == output_before, "Рефакторинг змінив поведінку звіту!"
print("✅ Вивід «до» і «після» повністю ідентичний — рефакторинг зберіг поведінку звіту.")

### Навіщо це все: звіт для інших даних — один виклик

Звіт за наступний вікенд тепер не потребує копіювання коду.

**Зміни:** додай до `next_weekend` ще один чек за неділю і передбач, який день стане найкращим, перш ніж запускати.

In [ ]:
next_weekend = [
    Order(610.0, 60.0, "сб", "обід", 2),
    Order(1200.0, 150.0, "нд", "вечеря", 6),
]
print_report(next_weekend)

## 🛠️ CREATE — самостійна декомпозиція

Дано плаский скрипт обрахунку бібліотечних штрафів. Розклади його на функції за тими самими трьома патернами:

- `is_overdue(days_late)` — **predicate**: чи є прострочення
- `calculate_fine(days_late, rate=FINE_PER_DAY)` — **transformer**: скільки грн штрафу за це позичення
- `total_fine(loans, rate=FINE_PER_DAY)` — **reducer**: загальний штраф по всіх позиченнях

In [ ]:
loans = [
    ("Кобзар", 0),
    ("1984", 5),
    ("Тіні забутих предків", 12),
    ("Захар Беркут", 0),
    ("Момент істини", 20),
]

FINE_PER_DAY = 2  # грн за день прострочення

# YOUR CODE HERE
# BEGIN SOLUTION
def is_overdue(days_late):
    return days_late > 0


def calculate_fine(days_late, rate=FINE_PER_DAY):
    return days_late * rate


def total_fine(loans, rate=FINE_PER_DAY):
    return sum(calculate_fine(days, rate) for _, days in loans if is_overdue(days))
# END SOLUTION

for title, days_late in loans:
    if is_overdue(days_late):
        print(f"{title}: прострочено на {days_late} дн., штраф {calculate_fine(days_late)} грн")
    else:
        print(f"{title}: без прострочення")

print(f"Загальний штраф: {total_fine(loans)} грн")

assert calculate_fine(5) == 10
assert calculate_fine(20) == 40
assert total_fine(loans) == 74
print("OK")

## 🔄 TRANSFER — та сама структура, інші дані

Той самий набір із трьох питань (predicate / transformer / reducer), але тепер про список треків плейлиста — **інша поверхнева деталь**, та сама структура рішення:

- `is_popular(plays)` — **predicate**: чи прослуховувань ≥ `MIN_POPULAR_PLAYS`
- `format_duration(seconds)` — **transformer**: секунди → рядок `"хв:сс"`
- підсумкова тривалість **популярних** треків — **reducer**

In [ ]:
tracks = [
    ("Ой у лузі червона калина", 187, 152000),
    ("Тримай", 203, 89000),
    ("Незалежність", 245, 310000),
    ("Спокій", 165, 12000),
    ("Гуцулка Ксеня", 198, 45000),
]  # (назва, тривалість_сек, кількість_прослуховувань)

MIN_POPULAR_PLAYS = 50000

# YOUR CODE HERE
# BEGIN SOLUTION
def is_popular(plays):
    return plays >= MIN_POPULAR_PLAYS


def format_duration(seconds):
    minutes, secs = divmod(seconds, 60)
    return f"{minutes}:{secs:02d}"


popular_tracks = [t for t in tracks if is_popular(t[2])]
total_popular_duration = sum(t[1] for t in popular_tracks)
# END SOLUTION

for title, duration, plays in tracks:
    marker = "★" if is_popular(plays) else " "
    print(f"{marker} {title:<30} {format_duration(duration)}  ({plays} прослуховувань)")

print()
print(f"Популярних треків: {len(popular_tracks)}")
print(f"Їхня сумарна тривалість: {format_duration(total_popular_duration)}")

assert format_duration(187) == "3:07"
assert [t[0] for t in popular_tracks] == ["Ой у лузі червона калина", "Тримай", "Незалежність"]
assert total_popular_duration == 635
assert format_duration(total_popular_duration) == "10:35"
print("OK")

## ✅ Самоперевірка (5 запитань)

**1.** Функція не має `return`. Що буде у змінній, якщо зберегти результат виклику цієї функції?

<details><summary>Відповідь</summary><code>None</code> — Python автоматично повертає <code>None</code>, якщо в тілі функції немає явного <code>return</code>.</details>

**2.** `def greet(name, greeting="Привіт"):` — що надрукує `greet("Оля")` і чим це відрізняється від `greet("Оля", "Вітаю")`?

<details><summary>Відповідь</summary><code>greet("Оля")</code> використовує значення параметра за замовчуванням і виведе те саме, що й із явним другим аргументом <code>"Привіт"</code>. <code>greet("Оля", "Вітаю")</code> перевизначає <code>greeting</code> переданим значенням.</details>

**3.** Функція `add_item(lst, x): lst.append(x); return lst` — чиста чи нечиста? Чому?

<details><summary>Відповідь</summary>Нечиста — вона <b>мутує</b> вхідний список <code>lst</code> (побічний ефект поза функцією), а не повертає новий список.</details>

**4.** Навіщо розбивати звіт кафе на `count_by_day`, `revenue_by_day`, `average_by_day`, `best_day`, `bills_by_time` і `print_report` замість одного суцільного блоку коду з уроку 6?

<details><summary>Відповідь</summary>Принцип єдиної відповідальності: кожна функція відповідає за одну чітку задачу (порахувати чеки, виторг, знайти лідера тощо), її легше назвати, перевірити окремо через <code>assert</code> і повторно використати — звіт для іншого списку чеків стає одним викликом <code>print_report(...)</code>. Змінні кожної функції локальні, тому не конфліктують між собою. А оскільки друкує лише <code>print_report</code>, решта функцій лишаються чистими.</details>

**5.** Задача: «з списку замовлень залишити тільки ті, що на суму більше 1000 грн». Це приклад predicate, transformer чи reducer?

<details><summary>Відповідь</summary>Predicate — для кожного елемента ставиться питання «так чи ні» (сума > 1000?), і за відповіддю елемент або залишається, або відкидається; кількість елементів на виході <b>менша або рівна</b> вхідній, форма самих елементів не змінюється.</details>

## Далі

**Урок 8 — Практикум П1. Big O + базові задачі.** Функції з цього уроку стають робочим інструментом: кожна задача практикуму — `fizzbuzz`, `is_palindrome`, `caesar_encode` / `caesar_decode` — це окрема функція з параметрами й `return`. Нове питання вже не лише «чи правильно вона працює», а й «скільки кроків вона робить»: `O(1)`, `O(n)`, `O(n²)`.

**Урок 9 — Декоратори.** Повертаємось до функцій на новому рівні: ім'я функції без дужок — це значення, яке можна передати в іншу функцію й повернути з неї. Там же з'являються `*args` / `**kwargs`.

Хочеш забігти наперед — повний конспект [`notes_functions.ipynb`](https://colab.research.google.com/github/NikoriakViktot/PY-Course-Victor-Nikoriak-22-09-2026/blob/main/module_1/lessons/lesson_07_functions/notes_functions.ipynb) лежить у цій же папці уроку.